# CS781 Capstone — Phase 3: Rhetorical Role Prediction

Name: Aditya Kumar | Roll No: 240059

In [ ]:
import os, re, gc, glob, math, json, time, random, shutil, zipfile, hashlib, subprocess, sys, csv
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_GPUS = torch.cuda.device_count()
print(DEVICE, NUM_GPUS)

In [ ]:
ARCH = "hybrid"                  # selected Phase-1 architecture
RUNS = [
    dict(method="lora"),
    dict(method="rslora"),
    dict(method="lorma"),
    dict(method="ia3"),
    dict(method="prompt"),
    dict(method="qlora"),
    dict(method="kd"),      # must come after at least one finished non-prompt run, because it uses the best one as its teacher
]
EXP_TAG = "p3"                   # prefix of every output name (keep the same across sessions)
SMOKE = False                    # True = tiny dry run (a few docs, 1 epoch) to check the pipeline before the long run

AUTO_ENSEMBLE = True             # rebuild the ensemble automatically at the end of every session
ENSEMBLE_ONLY = False            # True = skip training/Phase-1 files entirely, just ensemble the bundles (set RUNS = [])

hf_token = os.environ.get("HF_TOKEN")  # set HF_TOKEN before downloading Phase-3 data
HF_REPO, P3_DIR = "Exploration-Lab/CS-781-Capstone", "Phase_3_RR"
LOCAL_DATA_DIR = None                                     # set to a folder holding Train/ and Test/ to skip the download
INPUT_ROOT = os.environ.get("CS781_INPUT_ROOT", "/kaggle/input")   # where your Phase-1 zips/dataset are attached
OUT_ROOT = os.environ.get("CS781_OUT_ROOT", "/kaggle/working")

CONTEXT = "window"               # "window" = a sentence sees its neighbours (context-aware)   |   "sentence" = each sentence alone
WINDOW = 512                     # tokens per window (<= the Phase-1 model's max length)
MAX_SENT_TOK = 160               # a sentence is cut to this many tokens
USE_POS_FEATURE = True           # give the classifier the sentence's relative position in the document
BIDIRECTIONAL = True             # causal Phase-1 models: open the attention mask (tokens also see the right context)

EPOCHS = {"lora": 8, "rslora": 8, "lorma": 8, "ia3": 10, "prompt": 12, "qlora": 8, "kd": 8, "full": 6}
LR     = {"lora": 1e-3, "rslora": 5e-4, "lorma": 5e-4, "ia3": 3e-3, "prompt": 2e-2, "qlora": 1e-3, "kd": 2e-4, "full": 1e-4}
HEAD_LR, WEIGHT_DECAY, WARMUP_FRAC, PATIENCE = 2e-3, 0.01, 0.06, 3
BATCH_SIZE, GRAD_ACCUM = 8, 1
LABEL_BALANCE = "isqrt"          # "isqrt" = rare labels weighted up (mild) | "none"
BIAS_TUNE = True                 # after training, tune one logit bias per label on the dev set (macro-F1 under class imbalance)
RUN_MAX_MIN = None               # stop a run after the first epoch that ends beyond this many minutes (None = no limit)
PEFT = dict(r=16, alpha=32, rs_alpha=16, targets=("q_proj", "k_proj", "v_proj", "out_proj"),
            lorma_targets=("q_proj", "v_proj"), prompt_len=16)
KD_T, KD_ALPHA, STUDENT_KEEP_EVERY = 2.0, 0.5, 2         # student = every 2nd block of the Phase-1 model

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
WORK = os.path.join(OUT_ROOT, f"{EXP_TAG}_{ARCH}")
os.makedirs(WORK, exist_ok=True)
print("session:", EXP_TAG, ARCH, [r["method"] for r in RUNS])

In [ ]:
ROPE_BASE = 10000.0

# Architecture 1: decoder-only =====
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, max_seq_len, dropout=0.0):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.scale = 1.0 / math.sqrt(self.d_head)
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
        mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask", mask, persistent=False)

    def forward(self, x):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        scores = scores.masked_fill(~self.causal_mask[:T, :T], float("-inf"))
        attn = self.attn_dropout(torch.softmax(scores, dim=-1))
        out = torch.matmul(attn, v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.out_proj(out))


def sinusoidal_positional_encoding(max_len, d_model):
    pe = torch.zeros(max_len, d_model)
    position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model))
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe


class GPTBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, max_seq_len, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, max_seq_len, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model), nn.Dropout(dropout))

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x


class DecoderLM(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_heads, d_ff, max_seq_len, dropout=0.1):
        super().__init__()
        self.max_seq_len = max_seq_len
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.register_buffer("pos_emb", sinusoidal_positional_encoding(max_seq_len, d_model), persistent=False)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([GPTBlock(d_model, n_heads, d_ff, max_seq_len, dropout) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.token_emb.weight
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.drop(self.token_emb(idx) + self.pos_emb[:T].unsqueeze(0))
        for block in self.blocks:
            x = block(x)
        logits = self.lm_head(self.ln_f(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_p=0.95):
        self.eval()
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -self.max_seq_len:])
            logits = logits[:, -1, :] / max(temperature, 1e-5)
            probs = torch.softmax(logits, dim=-1)
            sorted_probs, sorted_idx = torch.sort(probs, descending=True, dim=-1)
            cum_probs = torch.cumsum(sorted_probs, dim=-1)
            cutoff = (cum_probs > top_p).float().argmax(dim=-1, keepdim=True)
            mask = torch.arange(sorted_probs.size(-1), device=idx.device).unsqueeze(0) > cutoff
            sorted_probs = sorted_probs.masked_fill(mask, 0.0)
            sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
            next_sorted = torch.multinomial(sorted_probs, num_samples=1)
            next_token = torch.gather(sorted_idx, -1, next_sorted)
            idx = torch.cat([idx, next_token], dim=1)
        return idx

# Architecture 2: encoder-decoder (encoder used) =====
class MultiHeadAttention(nn.Module):
    """Generic multi-head attention. causal=True masks future query->key positions (self-attn
    in the decoder). query and key/value can come from different sequences (cross-attention)."""
    def __init__(self, d_model, n_heads, dropout=0.0, causal=False, max_seq_len=None):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.scale = 1.0 / math.sqrt(self.d_head)
        self.causal = causal
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
        if causal:
            mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
            self.register_buffer("causal_mask", mask, persistent=False)

    def forward(self, x_q, x_kv):
        B, T, C = x_q.shape
        S = x_kv.shape[1]
        q = self.q_proj(x_q).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x_kv).view(B, S, self.n_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x_kv).view(B, S, self.n_heads, self.d_head).transpose(1, 2)
        scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        if self.causal:
            scores = scores.masked_fill(~self.causal_mask[:T, :S], float("-inf"))
        attn = self.attn_dropout(torch.softmax(scores, dim=-1))
        out = torch.matmul(attn, v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.out_proj(out))


def sinusoidal_positional_encoding(max_len, d_model):
    pe = torch.zeros(max_len, d_model)
    position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model))
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe


def feed_forward(d_model, d_ff, dropout):
    return nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model), nn.Dropout(dropout))


class EncoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout, causal=False)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = feed_forward(d_model, d_ff, dropout)

    def forward(self, x):
        x = x + self.self_attn(self.ln1(x), self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x


class DecoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, max_tgt_len, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout, causal=True, max_seq_len=max_tgt_len)
        self.ln2 = nn.LayerNorm(d_model)
        self.cross_attn = MultiHeadAttention(d_model, n_heads, dropout, causal=False)
        self.ln3 = nn.LayerNorm(d_model)
        self.ff = feed_forward(d_model, d_ff, dropout)

    def forward(self, x, enc_out):
        x = x + self.self_attn(self.ln1(x), self.ln1(x))
        x = x + self.cross_attn(self.ln2(x), enc_out)
        x = x + self.ff(self.ln3(x))
        return x


class EncoderDecoderLM(nn.Module):
    def __init__(self, vocab_size, d_model, n_enc_layers, n_dec_layers, n_heads, d_ff,
                 max_src_len, max_tgt_len, dropout=0.1):
        super().__init__()
        self.max_src_len = max_src_len
        self.max_tgt_len = max_tgt_len
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.register_buffer("enc_pos", sinusoidal_positional_encoding(max_src_len, d_model), persistent=False)
        self.register_buffer("dec_pos", sinusoidal_positional_encoding(max_tgt_len, d_model), persistent=False)
        self.drop = nn.Dropout(dropout)
        self.encoder_blocks = nn.ModuleList([EncoderBlock(d_model, n_heads, d_ff, dropout) for _ in range(n_enc_layers)])
        self.decoder_blocks = nn.ModuleList([DecoderBlock(d_model, n_heads, d_ff, max_tgt_len, dropout) for _ in range(n_dec_layers)])
        self.enc_ln_f = nn.LayerNorm(d_model)
        self.dec_ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.token_emb.weight
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)

    def encode(self, src_ids):
        T = src_ids.shape[1]
        x = self.drop(self.token_emb(src_ids) + self.enc_pos[:T].unsqueeze(0))
        for block in self.encoder_blocks:
            x = block(x)
        return self.enc_ln_f(x)

    def decode(self, tgt_input_ids, enc_out):
        T = tgt_input_ids.shape[1]
        x = self.drop(self.token_emb(tgt_input_ids) + self.dec_pos[:T].unsqueeze(0))
        for block in self.decoder_blocks:
            x = block(x, enc_out)
        return self.lm_head(self.dec_ln_f(x))

    def forward(self, src_ids, tgt_input_ids, tgt_target_ids=None):
        enc_out = self.encode(src_ids)
        logits = self.decode(tgt_input_ids, enc_out)
        loss = None
        if tgt_target_ids is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), tgt_target_ids.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, src_ids, bos_id, max_new_tokens, temperature=1.0, top_p=0.95):
        self.eval()
        enc_out = self.encode(src_ids[:, -self.max_src_len:])
        dec_ids = torch.full((src_ids.size(0), 1), bos_id, dtype=torch.long, device=src_ids.device)
        for _ in range(max_new_tokens):
            logits = self.decode(dec_ids[:, -self.max_tgt_len:], enc_out)
            logits = logits[:, -1, :] / max(temperature, 1e-5)
            probs = torch.softmax(logits, dim=-1)
            sorted_probs, sorted_idx = torch.sort(probs, descending=True, dim=-1)
            cum_probs = torch.cumsum(sorted_probs, dim=-1)
            cutoff = (cum_probs > top_p).float().argmax(dim=-1, keepdim=True)
            mask = torch.arange(sorted_probs.size(-1), device=src_ids.device).unsqueeze(0) > cutoff
            sorted_probs = sorted_probs.masked_fill(mask, 0.0)
            sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
            next_sorted = torch.multinomial(sorted_probs, num_samples=1)
            next_token = torch.gather(sorted_idx, -1, next_sorted)
            dec_ids = torch.cat([dec_ids, next_token], dim=1)
        return dec_ids[:, 1:]

# Architecture 3: hybrid prefix-LM =====
class PrefixAwareAttention(nn.Module):
    """Causal self-attention, except query positions inside the prefix (index < prefix_len)
    also get bidirectional access to the rest of the prefix. Positions >= prefix_len are
    plain causal. A single stack, single attention type per layer - neither a fully causal
    decoder nor a two-stack encoder-decoder."""
    def __init__(self, d_model, n_heads, max_seq_len, dropout=0.0):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.scale = 1.0 / math.sqrt(self.d_head)
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
        causal = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask", causal, persistent=False)

    def forward(self, x, prefix_len):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)

        pos = torch.arange(T, device=x.device)
        in_prefix = pos < prefix_len
        prefix_bonus = in_prefix.unsqueeze(1) & in_prefix.unsqueeze(0)
        allowed = self.causal_mask[:T, :T] | prefix_bonus

        scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        scores = scores.masked_fill(~allowed, float("-inf"))
        attn = self.attn_dropout(torch.softmax(scores, dim=-1))
        out = torch.matmul(attn, v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.out_proj(out))


def sinusoidal_positional_encoding(max_len, d_model):
    pe = torch.zeros(max_len, d_model)
    position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model))
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe


class PrefixLMBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, max_seq_len, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = PrefixAwareAttention(d_model, n_heads, max_seq_len, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model), nn.Dropout(dropout))

    def forward(self, x, prefix_len):
        x = x + self.attn(self.ln1(x), prefix_len)
        x = x + self.ff(self.ln2(x))
        return x


class PrefixLM(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_heads, d_ff, max_seq_len, dropout=0.1):
        super().__init__()
        self.max_seq_len = max_seq_len
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.register_buffer("pos_emb", sinusoidal_positional_encoding(max_seq_len, d_model), persistent=False)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([PrefixLMBlock(d_model, n_heads, d_ff, max_seq_len, dropout) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.token_emb.weight
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, idx, prefix_len, targets=None, loss_mask=None):
        B, T = idx.shape
        x = self.drop(self.token_emb(idx) + self.pos_emb[:T].unsqueeze(0))
        for block in self.blocks:
            x = block(x, prefix_len)
        logits = self.lm_head(self.ln_f(x))
        loss = None
        if targets is not None:
            per_tok = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.reshape(-1), reduction="none").view(B, T)
            m = (torch.arange(T, device=idx.device) + 1 >= prefix_len).to(per_tok.dtype)
            loss = (per_tok * m).sum() / (m.sum() * B).clamp(min=1.0)
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, prefix_len, max_new_tokens, temperature=1.0, top_p=0.95):
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.max_seq_len:]
            logits, _ = self(idx_cond, prefix_len=min(prefix_len, idx_cond.shape[1]))
            logits = logits[:, -1, :] / max(temperature, 1e-5)
            probs = torch.softmax(logits, dim=-1)
            sorted_probs, sorted_idx = torch.sort(probs, descending=True, dim=-1)
            cum_probs = torch.cumsum(sorted_probs, dim=-1)
            cutoff = (cum_probs > top_p).float().argmax(dim=-1, keepdim=True)
            mask = torch.arange(sorted_probs.size(-1), device=idx.device).unsqueeze(0) > cutoff
            sorted_probs = sorted_probs.masked_fill(mask, 0.0)
            sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
            next_sorted = torch.multinomial(sorted_probs, num_samples=1)
            next_token = torch.gather(sorted_idx, -1, next_sorted)
            idx = torch.cat([idx, next_token], dim=1)
        return idx

# Architecture 4: lightweight (RoPE/RMSNorm/SwiGLU/GQA) =====
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        norm = x * torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return norm * self.weight


def precompute_rope(d_head, max_seq_len, base=ROPE_BASE):
    inv_freq = 1.0 / (base ** (torch.arange(0, d_head, 2).float() / d_head))
    t = torch.arange(max_seq_len).float()
    freqs = torch.outer(t, inv_freq)
    return torch.cos(freqs), torch.sin(freqs)


def apply_rope(x, cos, sin):
    # x: (B, n_heads, T, d_head); cos/sin: (T, d_head/2)
    d = x.shape[-1]
    x1, x2 = x[..., : d // 2], x[..., d // 2 :]
    cos = cos.unsqueeze(0).unsqueeze(0)
    sin = sin.unsqueeze(0).unsqueeze(0)
    return torch.cat([x1 * cos - x2 * sin, x2 * cos + x1 * sin], dim=-1)


class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.gate_proj = nn.Linear(d_model, d_ff, bias=False)
        self.up_proj = nn.Linear(d_model, d_ff, bias=False)
        self.down_proj = nn.Linear(d_ff, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x)))


class GQAAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len, dropout=0.0):
        super().__init__()
        assert n_heads % n_kv_heads == 0
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.n_rep = n_heads // n_kv_heads
        self.d_head = d_model // n_heads
        self.scale = 1.0 / math.sqrt(self.d_head)
        self.q_proj = nn.Linear(d_model, n_heads * self.d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.out_proj = nn.Linear(n_heads * self.d_head, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
        causal = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask", causal, persistent=False)

    def forward(self, x, cos, sin):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)

        q = apply_rope(q, cos[:T], sin[:T])
        k = apply_rope(k, cos[:T], sin[:T])

        k = k.repeat_interleave(self.n_rep, dim=1)
        v = v.repeat_interleave(self.n_rep, dim=1)

        scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        scores = scores.masked_fill(~self.causal_mask[:T, :T], float("-inf"))
        attn = self.attn_dropout(torch.softmax(scores, dim=-1))
        out = torch.matmul(attn, v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.out_proj(out))


class LightBlock(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, d_ff, max_seq_len, dropout):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.attn = GQAAttention(d_model, n_heads, n_kv_heads, max_seq_len, dropout)
        self.norm2 = RMSNorm(d_model)
        self.ff = SwiGLU(d_model, d_ff, dropout)

    def forward(self, x, cos, sin):
        x = x + self.attn(self.norm1(x), cos, sin)
        x = x + self.ff(self.norm2(x))
        return x


class LightweightLM(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_heads, n_kv_heads, d_ff, max_seq_len, dropout=0.1):
        super().__init__()
        self.max_seq_len = max_seq_len
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            LightBlock(d_model, n_heads, n_kv_heads, d_ff, max_seq_len, dropout) for _ in range(n_layers)
        ])
        self.norm_f = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.token_emb.weight

        d_head = d_model // n_heads
        cos, sin = precompute_rope(d_head, max_seq_len)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.drop(self.token_emb(idx))
        for block in self.blocks:
            x = block(x, self.rope_cos, self.rope_sin)
        logits = self.lm_head(self.norm_f(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_p=0.95):
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.max_seq_len:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / max(temperature, 1e-5)
            probs = torch.softmax(logits, dim=-1)
            sorted_probs, sorted_idx = torch.sort(probs, descending=True, dim=-1)
            cum_probs = torch.cumsum(sorted_probs, dim=-1)
            cutoff = (cum_probs > top_p).float().argmax(dim=-1, keepdim=True)
            mask = torch.arange(sorted_probs.size(-1), device=idx.device).unsqueeze(0) > cutoff
            sorted_probs = sorted_probs.masked_fill(mask, 0.0)
            sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
            next_sorted = torch.multinomial(sorted_probs, num_samples=1)
            next_token = torch.gather(sorted_idx, -1, next_sorted)
            idx = torch.cat([idx, next_token], dim=1)
        return idx

# Architecture 5: RoPE + RMSNorm + SwiGLU + QK-Norm =====
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        norm = x * torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return norm * self.weight


def precompute_rope(d_head, max_seq_len, base=ROPE_BASE):
    inv_freq = 1.0 / (base ** (torch.arange(0, d_head, 2).float() / d_head))
    t = torch.arange(max_seq_len).float()
    freqs = torch.outer(t, inv_freq)
    return torch.cos(freqs), torch.sin(freqs)


def apply_rope(x, cos, sin):
    d = x.shape[-1]
    x1, x2 = x[..., : d // 2], x[..., d // 2 :]
    cos = cos.unsqueeze(0).unsqueeze(0)
    sin = sin.unsqueeze(0).unsqueeze(0)
    return torch.cat([x1 * cos - x2 * sin, x2 * cos + x1 * sin], dim=-1)


class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.gate_proj = nn.Linear(d_model, d_ff, bias=False)
        self.up_proj = nn.Linear(d_model, d_ff, bias=False)
        self.down_proj = nn.Linear(d_ff, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x)))


class QKNormAttention(nn.Module):
    """Full multi-head causal attention with RoPE, plus QK-Norm: Q and K are each RMSNorm'd
    per head (over the head dimension) before the dot product, bounding attention logit scale
    independent of how large Q/K activations get during training - improves stability."""
    def __init__(self, d_model, n_heads, max_seq_len, dropout=0.0):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.scale = 1.0 / math.sqrt(self.d_head)
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.q_norm = RMSNorm(self.d_head)
        self.k_norm = RMSNorm(self.d_head)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
        causal = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask", causal, persistent=False)

    def forward(self, x, cos, sin):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)

        q = self.q_norm(q)
        k = self.k_norm(k)
        q = apply_rope(q, cos[:T], sin[:T])
        k = apply_rope(k, cos[:T], sin[:T])

        scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        scores = scores.masked_fill(~self.causal_mask[:T, :T], float("-inf"))
        attn = self.attn_dropout(torch.softmax(scores, dim=-1))
        out = torch.matmul(attn, v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.out_proj(out))


class QKBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, max_seq_len, dropout):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.attn = QKNormAttention(d_model, n_heads, max_seq_len, dropout)
        self.norm2 = RMSNorm(d_model)
        self.ff = SwiGLU(d_model, d_ff, dropout)

    def forward(self, x, cos, sin):
        x = x + self.attn(self.norm1(x), cos, sin)
        x = x + self.ff(self.norm2(x))
        return x


class RopeSwiGLUQKNormLM(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_heads, d_ff, max_seq_len, dropout=0.1):
        super().__init__()
        self.max_seq_len = max_seq_len
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([QKBlock(d_model, n_heads, d_ff, max_seq_len, dropout) for _ in range(n_layers)])
        self.norm_f = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.token_emb.weight

        d_head = d_model // n_heads
        cos, sin = precompute_rope(d_head, max_seq_len)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.drop(self.token_emb(idx))
        for block in self.blocks:
            x = block(x, self.rope_cos, self.rope_sin)
        logits = self.lm_head(self.norm_f(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_p=0.95):
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.max_seq_len:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / max(temperature, 1e-5)
            probs = torch.softmax(logits, dim=-1)
            sorted_probs, sorted_idx = torch.sort(probs, descending=True, dim=-1)
            cum_probs = torch.cumsum(sorted_probs, dim=-1)
            cutoff = (cum_probs > top_p).float().argmax(dim=-1, keepdim=True)
            mask = torch.arange(sorted_probs.size(-1), device=idx.device).unsqueeze(0) > cutoff
            sorted_probs = sorted_probs.masked_fill(mask, 0.0)
            sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
            next_sorted = torch.multinomial(sorted_probs, num_samples=1)
            next_token = torch.gather(sorted_idx, -1, next_sorted)
            idx = torch.cat([idx, next_token], dim=1)
        return idx

In [ ]:
class BPE:
    """Phase-1 byte-level BPE (load + encode). Same merges, same special ids, same word/</s> convention as pretraining."""
    SPECIAL = ["<pad>", "<unk>", "<s>", "</s>"]
    def __init__(self):
        self.merges = {}; self.id_to_bytes = {i: bytes([i]) for i in range(256)}; self.special_to_id = {}
        for i, t in enumerate(self.SPECIAL):
            self.special_to_id[t] = 256 + i; self.id_to_bytes[256 + i] = t.encode()
    @staticmethod
    def _merge(seq, pair, new_id):
        out, i = [], 0
        while i < len(seq):
            if i < len(seq) - 1 and (seq[i], seq[i + 1]) == pair: out.append(new_id); i += 2
            else: out.append(seq[i]); i += 1
        return out
    @classmethod
    def load(cls, path):
        tok = cls()
        for ps, nid in json.load(open(path, encoding="utf-8"))["merges"].items():
            a, b = (int(x) for x in ps.split(","))
            tok.merges[(a, b)] = nid; tok.id_to_bytes[nid] = tok.id_to_bytes[a] + tok.id_to_bytes[b]
        return tok
    def encode_piece(self, piece):
        seq = list(piece.encode("utf-8"))
        while len(seq) > 1:
            cand = [p for p in zip(seq, seq[1:]) if p in self.merges]
            if not cand: break
            pair = min(cand, key=lambda p: self.merges[p]); seq = self._merge(seq, pair, self.merges[pair])
        return seq


class SentTok:
    def __init__(self, bpe):
        self.bpe, self.eos, self.pad, self.cache = bpe, bpe.special_to_id["</s>"], bpe.special_to_id["<pad>"], {}
    def encode(self, sent, max_tok):
        ids = []
        for w in str(sent).split():
            p = self.cache.get(w)
            if p is None: p = self.cache[w] = self.bpe.encode_piece(w) + [self.eos]
            ids.extend(p)
            if len(ids) >= max_tok: break
        return np.array(ids[:max_tok] or [self.eos], dtype=np.int64)


def read_json(p):
    with open(p, "r", encoding="utf-8") as f: return json.load(f)

def get_path(rel):
    if LOCAL_DATA_DIR: return os.path.join(LOCAL_DATA_DIR, rel)
    from huggingface_hub import hf_hub_download
    return hf_hub_download(repo_id=HF_REPO, filename=f"{P3_DIR}/{rel}", repo_type="dataset", token=hf_token)

def load_vocab(raw):
    """Same rules as the official evaluator: list, {label: id} or {id: label}."""
    if isinstance(raw, list): return [str(x) for x in raw]
    items = list(raw.items())
    if all(isinstance(v, int) for _, v in items): return [k for k, _ in sorted(items, key=lambda kv: kv[1])]
    return [v for _, v in sorted(items, key=lambda kv: int(kv[0]))]

def parse_docs(raw, domain, labelled):
    """Accepts [{id,text:[..],labels:[..]}] or {id: {text,labels}} or {id: [sentences]}."""
    pairs = list(raw.items()) if isinstance(raw, dict) else [(rec.get("id", rec.get("doc_id")), rec) for rec in raw]
    out = []
    for did, v in pairs:
        if isinstance(v, dict):
            sents = next((v[k] for k in ("text", "sentences", "sents") if k in v), None)
            labs = next((v[k] for k in ("labels", "label", "roles", "annotations") if k in v), None)
        else: sents, labs = v, None
        assert sents is not None, f"document {did}: no sentence list found (keys: {list(v) if isinstance(v, dict) else type(v)})"
        sents = [s if isinstance(s, str) else str(s.get("text", s)) for s in sents]
        if labelled:
            assert labs is not None and len(labs) == len(sents), f"document {did}: {len(sents)} sentences but labels={None if labs is None else len(labs)}"
        out.append(dict(id=str(did), domain=domain, sents=sents, labels=[str(l).strip() for l in labs] if labelled else None))
    return sorted(out, key=lambda d: d["id"])          # sorted -> identical order in every session (needed for ensembling)

vocab = load_vocab(read_json(get_path("Train/label_vocab.json")))
NL = len(vocab); L2I = {l: i for i, l in enumerate(vocab)}
DOCS = {"train": [], "dev": [], "test": []}
for dom in ("CL", "IT"):
    DOCS["train"] += parse_docs(read_json(get_path(f"Train/{dom}_train.json")), dom, True)
    DOCS["dev"] += parse_docs(read_json(get_path(f"Train/{dom}_dev.json")), dom, True)
    DOCS["test"] += parse_docs(read_json(get_path(f"Test/{dom}_test.json")), dom, False)
for sp in ("train", "dev"):
    bad = {l for d in DOCS[sp] for l in d["labels"] if l not in L2I}
    assert not bad, f"labels in {sp} that are not in label_vocab.json: {sorted(bad)[:10]}"
for sp in DOCS:
    ids = [d["id"] for d in DOCS[sp]]
    assert len(set(ids)) == len(ids), f"duplicate document ids inside {sp} (CL and IT must not share ids)"
if SMOKE:
    DOCS = {"train": DOCS["train"][:8] + [d for d in DOCS["train"] if d["domain"] == "IT"][:4], "dev": DOCS["dev"][:6] + [d for d in DOCS["dev"] if d["domain"] == "IT"][:3],
            "test": DOCS["test"][:4] + [d for d in DOCS["test"] if d["domain"] == "IT"][:3]}
    EPOCHS = {k: 1 for k in EPOCHS}; print(">>> SMOKE MODE: tiny subset, 1 epoch <<<")
for sp, ds in DOCS.items():
    print(f"{sp:5s}: {len(ds):4d} docs ({sum(d['domain']=='CL' for d in ds)} CL / {sum(d['domain']=='IT' for d in ds)} IT), {sum(len(d['sents']) for d in ds):7d} sentences")
cnt = np.bincount([L2I[l] for d in DOCS["train"] for l in d["labels"]], minlength=NL)
print("train label counts:", {vocab[i]: int(c) for i, c in enumerate(cnt)})

In [ ]:
CAUSAL_ARCHS = ("decoder", "lightweight", "rope_qknorm")

def find_phase1(arch):
    hits = glob.glob(os.path.join(INPUT_ROOT, "**", f"{arch}_model.pt"), recursive=True)
    assert hits, f"{arch}_model.pt not found under {INPUT_ROOT}. Attach the dataset made from your Phase-1 zips."
    d = os.path.dirname(hits[0])
    for n in (f"{arch}_bpe_tokenizer.json", "bpe_tokenizer.json"):
        if os.path.exists(os.path.join(d, n)): return hits[0], os.path.join(d, n)
    t = glob.glob(os.path.join(INPUT_ROOT, "**", f"{arch}_bpe_tokenizer.json"), recursive=True) or glob.glob(os.path.join(INPUT_ROOT, "**", "bpe_tokenizer.json"), recursive=True)
    assert t, "tokenizer json not found next to the model"
    return hits[0], t[0]

def load_phase1_backbone(arch, path):
    ck = torch.load(path, map_location="cpu", weights_only=False); cfg, sd = ck["config"], ck["state_dict"]; g = cfg.get
    if arch == "decoder": m = DecoderLM(g("vocab_size"), g("d_model"), g("n_layers"), g("n_heads"), g("d_ff"), g("max_seq_len"), g("dropout", 0.1))
    elif arch == "encdec": m = EncoderDecoderLM(g("vocab_size"), g("d_model"), g("n_enc_layers"), g("n_dec_layers"), g("n_heads"), g("d_ff"), g("max_src_len"), g("max_tgt_len"), g("dropout", 0.1))
    elif arch == "hybrid": m = PrefixLM(g("vocab_size"), g("d_model"), g("n_layers"), g("n_heads"), g("d_ff"), g("max_seq_len"), g("dropout", 0.1))
    elif arch == "lightweight": m = LightweightLM(g("vocab_size"), g("d_model"), g("n_layers"), g("n_heads"), g("n_kv_heads"), g("d_ff"), g("max_seq_len"), g("dropout", 0.1))
    elif arch == "rope_qknorm": m = RopeSwiGLUQKNormLM(g("vocab_size"), g("d_model"), g("n_layers"), g("n_heads"), g("d_ff"), g("max_seq_len"), g("dropout", 0.1))
    else: raise ValueError(arch)
    m.load_state_dict(sd, strict=True)
    return m, cfg, (g("max_seq_len") or g("max_src_len"))

def block_list_name(arch): return "encoder_blocks" if arch == "encdec" else "blocks"

def strip_unused(m, arch):
    m.lm_head = nn.Identity()
    if arch == "encdec": m.decoder_blocks = nn.ModuleList(); m.dec_ln_f = nn.Identity()

def prune_layers(m, arch, keep):
    blocks = getattr(m, block_list_name(arch)); setattr(m, block_list_name(arch), nn.ModuleList([blocks[i] for i in keep]))


class RRModel(nn.Module):
    """Phase-1 backbone -> token features -> mean-pool per sentence (+ relative position) -> label logits."""
    def __init__(self, arch, backbone, d_model, n_labels, bidirectional=True, use_pos=True, prompt_len=0):
        super().__init__()
        self.arch, self.backbone, self.bidir, self.use_pos, self.P, self.d_model = arch, backbone, bidirectional, use_pos, prompt_len, d_model
        self.prompt = nn.Parameter(torch.randn(prompt_len, d_model) * 0.02) if prompt_len else None
        self.head = nn.Sequential(nn.Dropout(0.1), nn.Linear(d_model + (2 if use_pos else 0), n_labels))
        if bidirectional and arch in CAUSAL_ARCHS:
            for mod in backbone.modules():
                if hasattr(mod, "causal_mask"): mod.causal_mask.fill_(True)

    def token_features(self, ids):
        m = self.backbone; e = m.token_emb(ids)
        if self.prompt is not None: e = torch.cat([self.prompt.unsqueeze(0).expand(e.shape[0], -1, -1).to(e.dtype), e], 1)
        T = e.shape[1]
        if self.arch == "decoder":
            x = m.drop(e + m.pos_emb[:T].unsqueeze(0))
            for b in m.blocks: x = b(x)
            x = m.ln_f(x)
        elif self.arch == "encdec":
            x = m.drop(e + m.enc_pos[:T].unsqueeze(0))
            for b in m.encoder_blocks: x = b(x)
            x = m.enc_ln_f(x)
        elif self.arch == "hybrid":
            x = m.drop(e + m.pos_emb[:T].unsqueeze(0)); plen = T if self.bidir else 0
            for b in m.blocks: x = b(x, plen)
            x = m.ln_f(x)
        else:
            x = m.drop(e)
            for b in m.blocks: x = b(x, m.rope_cos, m.rope_sin)
            x = m.norm_f(x)
        return x[:, self.P:]

    def forward(self, ids, sidx, rel):
        h = self.token_features(ids).float(); B, T, D = h.shape; S = rel.shape[1]
        idx = sidx.clone(); idx[idx < 0] = S                                  # padding -> dummy slot S
        sums = torch.zeros(B, S + 1, D, device=h.device).scatter_add_(1, idx.unsqueeze(-1).expand(-1, -1, D), h)
        cnt = torch.zeros(B, S + 1, device=h.device).scatter_add_(1, idx, torch.ones_like(idx, dtype=torch.float))
        pooled = (sums / cnt.clamp(min=1).unsqueeze(-1))[:, :S]
        if self.use_pos: pooled = torch.cat([pooled, rel.to(pooled.dtype)], -1)
        return self.head(pooled.to(self.head[1].weight.dtype))

In [ ]:
class LoRALinear(nn.Module):
    """LoRA: y = Wx + s*B(Ax). rsLoRA = same with s = alpha/sqrt(r) instead of alpha/r (rank-stabilised)."""
    def __init__(self, base, r, alpha, rs=False):
        super().__init__()
        self.base, self.s = base, (alpha / math.sqrt(r) if rs else alpha / r)
        self.A = nn.Parameter(torch.empty(r, base.in_features)); self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
    def forward(self, x): return self.base(x) + self.s * F.linear(F.linear(x, self.A), self.B)

class LoRMALinear(nn.Module):
    """LoRMA+ : W' = (I + (alpha/r) B A) W0 (multiplicative); A=0, B=kaiming as in the authors' code, so W'=W0 at step 0."""
    def __init__(self, base, r, alpha):
        super().__init__()
        self.base, self.s = base, alpha / r
        self.A = nn.Parameter(torch.zeros(r, base.out_features)); self.B = nn.Parameter(torch.empty(base.out_features, r))
        nn.init.kaiming_uniform_(self.B, a=math.sqrt(5))
    def forward(self, x):
        h = F.linear(x, self.base.weight); y = h + self.s * F.linear(F.linear(h, self.A), self.B)
        return y + self.base.bias if self.base.bias is not None else y

class IA3Linear(nn.Module):
    """IA3: the output of a (frozen) linear layer is rescaled by a learned vector (init 1)."""
    def __init__(self, base):
        super().__init__(); self.base = base; self.vec = nn.Parameter(torch.ones(base.out_features))
    def forward(self, x): return self.base(x) * self.vec.to(x.dtype)

class IA3Scale(nn.Module):
    def __init__(self, d): super().__init__(); self.vec = nn.Parameter(torch.ones(d))
    def forward(self, x): return x * self.vec.to(x.dtype)

NF4 = [-1.0, -0.6961928009986877, -0.5250730514526367, -0.39491748809814453, -0.28444138169288635, -0.18477343022823334,
       -0.09105003625154495, 0.0, 0.07958029955625534, 0.16093020141124725, 0.24611230194568634, 0.33791524171829224,
       0.44070982933044434, 0.5626170039176941, 0.7229568362236023, 1.0]

class QLoRALinear(nn.Module):
    """QLoRA: frozen base weight stored as block-wise 4-bit NF4 (two codes per byte, block=64, absmax scale) + trainable LoRA."""
    def __init__(self, base, r, alpha, block=64):
        super().__init__()
        w = base.weight.detach().float().flatten(); self.n, self.shape, self.block = w.numel(), tuple(base.weight.shape), block
        if (-self.n) % block: w = F.pad(w, (0, (-self.n) % block))
        blocks = w.view(-1, block); absmax = blocks.abs().max(1, keepdim=True).values.clamp(min=1e-8)
        table = torch.tensor(NF4, device=w.device)
        codes = ((blocks / absmax).unsqueeze(-1) - table).abs().argmin(-1).to(torch.uint8)
        self.register_buffer("packed", (codes[:, 0::2] << 4) | codes[:, 1::2]); self.register_buffer("absmax", absmax.squeeze(1))
        self.register_buffer("table", table)
        self.register_buffer("bias", None if base.bias is None else base.bias.detach().clone())
        self.s = alpha / r; self.in_features, self.out_features = base.in_features, base.out_features
        self.A = nn.Parameter(torch.empty(r, base.in_features)); self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
    def dequant(self):
        codes = torch.stack([(self.packed >> 4).long(), (self.packed & 0xF).long()], -1).view(self.packed.shape[0], -1)
        return (self.table[codes] * self.absmax[:, None]).flatten()[:self.n].view(self.shape)
    def forward(self, x):
        return F.linear(x, self.dequant().to(x.dtype), None if self.bias is None else self.bias.to(x.dtype)) + self.s * F.linear(F.linear(x, self.A), self.B)


def _swap(root, names, make):
    n = 0
    for _, mod in list(root.named_modules()):
        for cname, child in list(mod.named_children()):
            if isinstance(child, nn.Linear) and cname in names: setattr(mod, cname, make(child)); n += 1
    return n

def apply_method(model, method):
    bb = model.backbone; P = PEFT
    if method in ("full", "kd"):
        for p in model.parameters(): p.requires_grad = True
    else:
        for p in bb.parameters(): p.requires_grad = False
        if method == "lora": n = _swap(bb, P["targets"], lambda b: LoRALinear(b, P["r"], P["alpha"]))
        elif method == "rslora": n = _swap(bb, P["targets"], lambda b: LoRALinear(b, P["r"], P["rs_alpha"], rs=True))
        elif method == "lorma": n = _swap(bb, P["lorma_targets"], lambda b: LoRMALinear(b, P["r"], P["alpha"]))
        elif method == "qlora": n = _swap(bb, P["targets"], lambda b: QLoRALinear(b, P["r"], P["alpha"]))
        elif method == "ia3":
            n = _swap(bb, ("k_proj", "v_proj"), IA3Linear)
            for _, mod in list(bb.named_modules()):
                if type(mod).__name__ == "SwiGLU": mod.up_proj = IA3Linear(mod.up_proj); n += 1
                for cname, ch in list(mod.named_children()):
                    if cname == "ff" and isinstance(ch, nn.Sequential) and len(ch) == 4 and isinstance(ch[1], nn.GELU):
                        ch[1] = nn.Sequential(nn.GELU(), IA3Scale(ch[0].out_features)); n += 1
        elif method == "prompt": n = 1
        else: raise ValueError(f"unknown method {method}")
        assert n > 0, f"{method}: nothing to adapt"
    for p in model.head.parameters(): p.requires_grad = True
    if model.prompt is not None: model.prompt.requires_grad = True
    tr = sum(p.numel() for p in model.parameters() if p.requires_grad); tot = sum(p.numel() for p in model.parameters())
    print(f"[{method}] trainable {tr/1e6:.3f}M of {tot/1e6:.2f}M ({100*tr/tot:.2f}%)")
    return tr, tot

def trainable_state(model):
    names = {n for n, p in model.named_parameters() if p.requires_grad}
    return {k: v.detach().cpu() for k, v in model.state_dict().items() if k in names}

In [ ]:
def build_windows(lens, Wtok, mode):
    n = len(lens)
    if mode == "sentence": return [(i, i + 1) for i in range(n)]
    wins, i = [], 0
    while i < n:
        j, tot = i, 0
        while j < n and (j == i or tot + lens[j] <= Wtok): tot += lens[j]; j += 1
        wins.append((i, j))
        if j >= n: break
        i += max(1, (j - i) // 2)                                   # 50% overlap between consecutive windows
    return wins

def pack(docs, toks, Wtok, mode, pad_id):
    Wlen = MAX_SENT_TOK if mode == "sentence" else Wtok
    assert MAX_SENT_TOK <= Wlen, "MAX_SENT_TOK must fit in a window"
    rows = [(d, i, j) for d, st in enumerate(toks) for (i, j) in build_windows([len(x) for x in st], Wtok, mode)]
    N, S = len(rows), max(j - i for _, i, j in rows)
    X = np.full((N, Wlen), pad_id, np.int64); SI = np.full((N, Wlen), -1, np.int64)
    Y = np.full((N, S), -100, np.int64); R = np.zeros((N, S, 2), np.float32)
    for r, (d, i, j) in enumerate(rows):
        st, pos, n = toks[d], 0, len(toks[d])
        for k in range(i, j):
            a = st[k]; X[r, pos:pos + len(a)] = a; SI[r, pos:pos + len(a)] = k - i; pos += len(a)
        if docs[d]["labels"] is not None: Y[r, :j - i] = [L2I[l] for l in docs[d]["labels"][i:j]]
        rel = np.arange(i, j) / max(n - 1, 1); R[r, :j - i, 0] = rel; R[r, :j - i, 1] = rel ** 2
    return dict(X=torch.from_numpy(X), SI=torch.from_numpy(SI), Y=torch.from_numpy(Y), R=torch.from_numpy(R), rows=rows,
                n_sents=[len(t) for t in toks])

@torch.no_grad()
def predict_logp(model, pk, bs=16):
    """Per-sentence log-probs; a sentence covered by several windows gets a centre-weighted average."""
    model.eval(); rows = pk["rows"]
    acc = [np.zeros((n, NL), np.float32) for n in pk["n_sents"]]; ws = [np.zeros(n, np.float32) for n in pk["n_sents"]]
    for b in range(0, len(rows), bs):
        with torch.autocast(device_type="cuda", enabled=DEVICE.type == "cuda"):
            lg = model(pk["X"][b:b + bs].to(DEVICE), pk["SI"][b:b + bs].to(DEVICE), pk["R"][b:b + bs].to(DEVICE))
        lp = F.log_softmax(lg.float(), -1).cpu().numpy()
        for t, (d, i, j) in enumerate(rows[b:b + bs]):
            m = j - i; c = (m - 1) / 2; w = (1.0 - 0.9 * np.abs(np.arange(m) - c) / max(c, 1)).astype(np.float32)
            acc[d][i:j] += lp[t, :m] * w[:, None]; ws[d][i:j] += w
    return [a / w[:, None] for a, w in zip(acc, ws)]

def macro_f1(y, p):
    cm = np.bincount(y * NL + p, minlength=NL * NL).reshape(NL, NL)
    tp = np.diag(cm); fp = cm.sum(0) - tp; fn = cm.sum(1) - tp; den = 2 * tp + fp + fn
    f1 = np.where(den > 0, 2 * tp / np.maximum(den, 1), 0.0)       # == sklearn macro-F1 over all labels, zero_division=0
    return float(f1.mean()), f1

def flat_labels(docs): return np.array([L2I[l] for d in docs for l in d["labels"]], dtype=np.int64)
def flat_logp(lps): return np.concatenate(lps, 0)

def tune_bias(lp, y, grid=np.arange(-3, 3.01, 0.25), passes=2, min_gain=0.002):
    b = np.zeros(NL); base = macro_f1(y, lp.argmax(1))[0]; best = base
    for _ in range(passes):
        for k in range(NL):
            bk, bs = b[k], best
            for g in grid:
                b[k] = g; s = macro_f1(y, (lp + b).argmax(1))[0]
                if s > bs + 1e-9 or (abs(s - bs) <= 1e-9 and abs(g) < abs(bk)): bs, bk = s, g
            b[k], best = bk, bs
    return (b, best) if best - base >= min_gain else (np.zeros(NL), base)

def split_by_doc(flat, docs):
    out, o = [], 0
    for d in docs: out.append(flat[o:o + len(d["sents"])]); o += len(d["sents"])
    return out

In [ ]:
def build_rr(arch, p1_path, method, keep_every=None):
    bb, cfg, max_len = load_phase1_backbone(arch, p1_path); strip_unused(bb, arch)
    if keep_every: prune_layers(bb, arch, list(range(0, len(getattr(bb, block_list_name(arch))), keep_every)))
    P = PEFT["prompt_len"] if method == "prompt" else 0
    model = RRModel(arch, bb, cfg["d_model"], NL, BIDIRECTIONAL, USE_POS_FEATURE, P)
    apply_method(model, method)
    return model.to(DEVICE), cfg, max_len, P

def loss_fn(lg, y, w, t_lg=None):
    m = y != -100; s, yy = lg[m].float(), y[m]
    hard = F.cross_entropy(s, yy, weight=w)
    if t_lg is None: return hard
    soft = F.kl_div(F.log_softmax(s / KD_T, -1), F.softmax(t_lg[m].float() / KD_T, -1), reduction="batchmean") * KD_T * KD_T
    return KD_ALPHA * hard + (1 - KD_ALPHA) * soft

def fit(model, pk, eval_fn, ckdir, method, class_w, teacher=None, seed=0):
    os.makedirs(ckdir, exist_ok=True); last_p, best_p = f"{ckdir}/last.pt", f"{ckdir}/best.pt"
    dm = nn.DataParallel(model) if NUM_GPUS > 1 else model
    head_ids = {id(p) for p in model.head.parameters()}; prompt_ids = {id(model.prompt)} if model.prompt is not None else set()
    tr = [p for p in model.parameters() if p.requires_grad]
    groups = [{"params": [p for p in tr if id(p) in head_ids], "lr": HEAD_LR},
              {"params": [p for p in tr if id(p) not in head_ids], "lr": LR[method]}]
    opt = torch.optim.AdamW([g for g in groups if g["params"]], weight_decay=WEIGHT_DECAY)
    base = [g["lr"] for g in opt.param_groups]; scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")
    ds = torch.utils.data.TensorDataset(pk["X"], pk["SI"], pk["R"], pk["Y"])
    loader = torch.utils.data.DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=len(ds) > BATCH_SIZE, generator=torch.Generator().manual_seed(seed))
    epochs = EPOCHS[method]; spe = max(len(loader) // GRAD_ACCUM, 1); total, warm = epochs * spe, max(int(WARMUP_FRAC * epochs * spe), 1)
    cw = class_w.to(DEVICE); start, best, hist, bad = 0, -1.0, [], 0
    if os.path.exists(last_p):
        ck = torch.load(last_p, map_location="cpu", weights_only=False)
        model.load_state_dict(ck["trainable"], strict=False); opt.load_state_dict(ck["opt"]); scaler.load_state_dict(ck["scaler"])
        start, best, hist, bad = ck["epoch"], ck["best"], ck["hist"], ck["bad"]; print(f"resumed at epoch {start}/{epochs}")
    step, t0 = start * spe, time.time()
    for ep in range(start, epochs):
        dm.train(); run = nb = 0; opt.zero_grad(set_to_none=True)
        for i, (x, si, r, y) in enumerate(loader):
            x, si, r, y = x.to(DEVICE), si.to(DEVICE), r.to(DEVICE), y.to(DEVICE)
            with torch.autocast(device_type="cuda", enabled=DEVICE.type == "cuda"):
                lg = dm(x, si, r)
                tl = None
                if teacher is not None:
                    with torch.no_grad(): tl = teacher(x, si, r)
                loss = loss_fn(lg, y, cw, tl) / GRAD_ACCUM
            scaler.scale(loss).backward(); run += loss.item() * GRAD_ACCUM; nb += 1
            if (i + 1) % GRAD_ACCUM == 0:
                f = (step + 1) / warm if step < warm else 0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * (step - warm) / max(total - warm, 1)))
                for g, b in zip(opt.param_groups, base): g["lr"] = b * f
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(tr, 1.0); scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); step += 1
        f1 = eval_fn(model); hist.append(dict(epoch=ep + 1, loss=run / max(nb, 1), dev_macro_f1=f1))
        print(f"  epoch {ep+1}/{epochs} | loss {run/max(nb,1):.4f} | dev macro-F1 {f1:.4f} | {(time.time()-t0)/60:.1f} min", flush=True)
        if f1 > best: best, bad = f1, 0; torch.save({"trainable": trainable_state(model)}, best_p + ".tmp"); os.replace(best_p + ".tmp", best_p)
        else: bad += 1
        torch.save({"trainable": trainable_state(model), "opt": opt.state_dict(), "scaler": scaler.state_dict(), "epoch": ep + 1, "best": best, "hist": hist, "bad": bad}, last_p + ".tmp"); os.replace(last_p + ".tmp", last_p)
        if bad >= PATIENCE: print("  early stop (patience)"); break
        if RUN_MAX_MIN and (time.time() - t0) / 60 > RUN_MAX_MIN: print("  stopping: time limit for this run"); break
    model.load_state_dict(torch.load(best_p, map_location="cpu", weights_only=False)["trainable"], strict=False)
    return hist, best

In [ ]:
if ENSEMBLE_ONLY:
    RUNS = []; print("ENSEMBLE_ONLY: skipping Phase-1 model / tokenizer / training")
else:
    P1_MODEL, P1_TOK = find_phase1(ARCH)
    tokenizer = SentTok(BPE.load(P1_TOK)); TOK_SHA = hashlib.sha256(open(P1_TOK, "rb").read()).hexdigest()
    print("phase-1 model:", P1_MODEL, "| tokenizer sha:", TOK_SHA[:12])
    TOKS = {sp: [[tokenizer.encode(s, MAX_SENT_TOK) for s in d["sents"]] for d in DOCS[sp]] for sp in DOCS}
    ntok = [len(a) for sp in ("train",) for d in TOKS[sp] for a in d]
    print(f"tokens/sentence: mean {np.mean(ntok):.1f}, p95 {int(np.percentile(ntok, 95))} (cut at {MAX_SENT_TOK}) | docs: {len(TOKS['train'])} train")
Y_DEV = flat_labels(DOCS["dev"]); PACKS = {}
def get_packs(Wtok):
    if Wtok not in PACKS: PACKS[Wtok] = {sp: pack(DOCS[sp], TOKS[sp], Wtok, CONTEXT, tokenizer.pad) for sp in DOCS}
    return PACKS[Wtok]
def domain_f1(flat_pred):
    out = {}; o = 0
    for dom in ("CL", "IT"):
        idx = np.concatenate([np.arange(o + sum(len(x["sents"]) for x in DOCS["dev"][:k]), o + sum(len(x["sents"]) for x in DOCS["dev"][:k + 1])) for k, d in enumerate(DOCS["dev"]) if d["domain"] == dom] or [np.array([], int)])
        out[dom] = macro_f1(Y_DEV[idx], flat_pred[idx])[0] if len(idx) else float("nan")
    return out
def save_logp(path, lps): np.savez(path, **{f"d{i}": a.astype(np.float16) for i, a in enumerate(lps)})
def load_logp(path): z = np.load(path); return [z[f"d{i}"].astype(np.float32) for i in range(len(z.files))]
def write_submission(pred_flat, name, outdir):
    per = split_by_doc(pred_flat, DOCS["test"]); sub = {d["id"]: [vocab[k] for k in p] for d, p in zip(DOCS["test"], per)}
    for d in DOCS["test"]: assert len(sub[d["id"]]) == len(d["sents"])
    pj = f"{outdir}/prediction.json"; json.dump(sub, open(pj, "w"))
    zp = f"{OUT_ROOT}/{name}_submission.zip"
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z: z.write(pj, arcname="prediction.json")
    return zp
def run_official_eval(dev_pred_flat, outdir):
    sc = glob.glob(os.path.join(INPUT_ROOT, "**", "evaluate_phase3.py"), recursive=True)
    if not sc: return None
    ids = lambda dom: [d for d in DOCS["dev"] if d["domain"] == dom]
    for dom in ("CL", "IT"): json.dump({d["id"]: d["labels"] for d in ids(dom)}, open(f"{outdir}/dev_truth_{dom.lower()}.json", "w"))
    json.dump(vocab, open(f"{outdir}/labels.json", "w"))
    json.dump({d["id"]: [vocab[k] for k in p] for d, p in zip(DOCS["dev"], split_by_doc(dev_pred_flat, DOCS["dev"]))}, open(f"{outdir}/dev_pred.json", "w"))
    r = subprocess.run([sys.executable, sc[0], "--truth-cl", f"{outdir}/dev_truth_cl.json", "--truth-it", f"{outdir}/dev_truth_it.json",
                        "--prediction", f"{outdir}/dev_pred.json", "--labels", f"{outdir}/labels.json"], capture_output=True, text=True)
    return (r.stdout + r.stderr).strip()

def run_one(spec):
    method = spec["method"]; name = spec.get("name", method); outdir = f"{WORK}/{name}"; os.makedirs(outdir, exist_ok=True)
    if os.path.exists(f"{outdir}/done.json"): print(f"[{name}] already done -> skipped"); return json.load(open(f"{outdir}/done.json"))
    print(f"\n{'='*20} {ARCH} / {name} {'='*20}"); t_start = time.time()
    model, cfg, max_len, P = build_rr(ARCH, P1_MODEL, method, STUDENT_KEEP_EVERY if method == "kd" else None)
    Wtok = WINDOW - P; assert Wtok + P <= max_len, f"WINDOW {WINDOW} exceeds the Phase-1 max length {max_len}"
    pk = get_packs(Wtok)
    cnt = np.bincount(pk["train"]["Y"][pk["train"]["Y"] >= 0].numpy(), minlength=NL).astype(np.float64)
    w = np.ones(NL)
    if LABEL_BALANCE == "isqrt": w = np.clip(np.sqrt(np.median(cnt[cnt > 0]) / np.maximum(cnt, 1)), 0.5, 4.0); w[cnt == 0] = 1.0
    teacher = None
    if method == "kd":
        cands = [m for m in (json.load(open(p)) for p in glob.glob(f"{WORK}/*/done.json")) if m["method"] not in ("kd", "prompt")]
        assert cands, "kd needs a finished non-prompt run of the same architecture earlier in RUNS"
        tm = max(cands, key=lambda m: m["dev_f1_raw"]); print(f"  teacher = {tm['name']} (dev macro-F1 {tm['dev_f1_raw']:.4f}); student keeps every {STUDENT_KEEP_EVERY}nd block")
        teacher, _, _, _ = build_rr(ARCH, P1_MODEL, tm["method"])
        teacher.load_state_dict(torch.load(f"{WORK}/{tm['name']}/ckpt/best.pt", map_location="cpu", weights_only=False)["trainable"], strict=False)
        teacher.eval(); [p.requires_grad_(False) for p in teacher.parameters()]
    ev = lambda m: macro_f1(Y_DEV, flat_logp(predict_logp(m, pk["dev"])).argmax(1))[0]
    hist, best = fit(model, pk["train"], ev, f"{outdir}/ckpt", method, torch.tensor(w, dtype=torch.float32), teacher)
    lp_dev, lp_test = predict_logp(model, pk["dev"]), predict_logp(model, pk["test"])
    save_logp(f"{outdir}/lp_dev.npz", lp_dev); save_logp(f"{outdir}/lp_test.npz", lp_test)
    fd, ft = flat_logp(lp_dev), flat_logp(lp_test); raw = macro_f1(Y_DEV, fd.argmax(1))
    b, tuned = tune_bias(fd, Y_DEV) if BIAS_TUNE else (np.zeros(NL), raw[0])
    pred_dev, pred_test = (fd + b).argmax(1), (ft + b).argmax(1)
    meta = dict(task="phase3_rr", name=name, arch=ARCH, method=method, tag=EXP_TAG, tokenizer_sha=TOK_SHA, doc_ids_dev=[d["id"] for d in DOCS["dev"]], doc_ids_test=[d["id"] for d in DOCS["test"]],
                dev_f1_raw=raw[0], dev_f1_tuned=tuned, dev_f1_by_domain=domain_f1(pred_dev), per_label_f1={vocab[i]: float(f) for i, f in enumerate(raw[1])}, bias=b.tolist(), context=CONTEXT,
                trainable_M=round(sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6, 3), total_M=round(sum(p.numel() for p in model.parameters()) / 1e6, 2),
                lr=LR[method], epochs_run=len(hist), minutes=round((time.time() - t_start) / 60, 1), smoke=SMOKE)
    json.dump(meta, open(f"{outdir}/meta.json", "w"), indent=1)
    zp = write_submission(pred_test, f"{EXP_TAG}_{ARCH}_{name}", outdir)
    off = run_official_eval(pred_dev, outdir)
    print(f"  dev macro-F1: raw {raw[0]:.4f} -> bias-tuned {tuned:.4f} | CL {meta['dev_f1_by_domain']['CL']:.4f} IT {meta['dev_f1_by_domain']['IT']:.4f}\n  submission: {zp}")
    if off: print("  official evaluator says:\n   " + off.replace("\n", "\n   "))
    json.dump(meta, open(f"{outdir}/done.json", "w"))
    del model, teacher; gc.collect(); torch.cuda.empty_cache()
    return meta

RESULTS = [run_one(s) for s in RUNS]
print("\nSESSION SUMMARY"); [print(f"  {m['name']:12s} raw {m['dev_f1_raw']:.4f} | tuned {m['dev_f1_tuned']:.4f} | {m['minutes']} min") for m in RESULTS]

In [ ]:
def collect_bundles():
    """Unpack every *_bundle.zip (attached inputs + this session's) under OUT_ROOT/_bundles."""
    dest = f"{OUT_ROOT}/_bundles"; os.makedirs(dest, exist_ok=True); zips = []
    zips += glob.glob(os.path.join(INPUT_ROOT, "**", "*_bundle.zip"), recursive=True) + glob.glob(f"{OUT_ROOT}/*_bundle.zip")
    for zp in zips:
        try:
            with zipfile.ZipFile(zp) as z: z.extractall(f"{dest}/{os.path.basename(zp)[:-4]}")
        except Exception as e: print("could not unpack", zp, repr(e)[:100])

def run_ensemble():
    collect_bundles()
    metas = []
    for root in (OUT_ROOT, INPUT_ROOT):
        for mp in glob.glob(os.path.join(root, "**", "meta.json"), recursive=True):
            try: m = json.load(open(mp))
            except Exception: continue
            d = os.path.dirname(mp)
            if m.get("task") == "phase3_rr" and not m.get("smoke") and os.path.exists(os.path.join(d, "lp_dev.npz")):
                m["_dir"], m["_key"], m["_t"] = d, (m["arch"], m["name"]), os.path.getmtime(os.path.join(d, "lp_dev.npz")); metas.append(m)
    seen = {}
    for m in sorted(metas, key=lambda m: -m["_t"]): seen.setdefault(m["_key"], m)     # same run found twice -> keep the newest copy
    metas = sorted(seen.values(), key=lambda m: -m["dev_f1_raw"])
    if not metas: print("no finished (non-smoke) runs found - nothing to ensemble yet"); return
    ref = metas[0]
    metas = [m for m in metas if m["doc_ids_dev"] == ref["doc_ids_dev"] and m["doc_ids_test"] == ref["doc_ids_test"]]
    FD = [flat_logp(load_logp(m["_dir"] + "/lp_dev.npz")) for m in metas]; FT = [flat_logp(load_logp(m["_dir"] + "/lp_test.npz")) for m in metas]
    print(f"{len(metas)} runs in the ensemble pool:"); [print(f"  {m['arch']:11s} {m['name']:8s} dev macro-F1 {m['dev_f1_raw']:.4f}") for m in metas]

    def mix(arrs, idx):                                              # average of probabilities, returned as log-probs
        s = np.stack([arrs[i] for i in idx]); mx = s.max(0)
        return mx + np.log(np.exp(s - mx).mean(0))
    def finish(idx, name):
        fd, ft = mix(FD, idx), mix(FT, idx); raw = macro_f1(Y_DEV, fd.argmax(1))[0]
        b, tuned = tune_bias(fd, Y_DEV) if BIAS_TUNE else (np.zeros(NL), raw)
        od = f"{OUT_ROOT}/{EXP_TAG}_ensemble_{name}"; os.makedirs(od, exist_ok=True)
        zp = write_submission((ft + b).argmax(1), f"{EXP_TAG}_ensemble_{name}", od)
        print(f"  {name:8s} members={len(idx)} dev macro-F1 raw {raw:.4f} | bias-tuned {tuned:.4f} -> {os.path.basename(zp)}")
        return tuned, zp, name
    cands = [finish(list(range(len(metas))), "all"), finish(list(range(min(4, len(metas)))), "top4")]
    sel, best = [0], macro_f1(Y_DEV, FD[0].argmax(1))[0]             # greedy forward selection on dev (optimistic: chosen on dev)
    while True:
        gains = [(macro_f1(Y_DEV, mix(FD, sel + [i]).argmax(1))[0], i) for i in range(len(metas)) if i not in sel]
        if not gains or max(gains)[0] <= best + 1e-4: break
        best, nxt = max(gains); sel.append(nxt)
    cands.append(finish(sel, "greedy"))
    tuned, zp, name = max(cands, key=lambda c: c[0]); final = f"{OUT_ROOT}/{EXP_TAG}_FINAL_submission.zip"; shutil.copyfile(zp, final)
    print(f"\n>>> FINAL = '{name}' ensemble (dev macro-F1 {tuned:.4f}) -> {final}\n    Submit this one file. (Dev scores are measured on dev; test may differ.)")
    return final

In [ ]:
def zip_dir(zpath, files):
    with zipfile.ZipFile(zpath, "w", zipfile.ZIP_STORED) as z:
        for f, arc in files:
            if os.path.exists(f): z.write(f, arc)
    print(f"{os.path.getsize(zpath)/1e6:.1f} MB  {zpath}")
runs = [os.path.dirname(p) for p in glob.glob(f"{WORK}/*/done.json")]
bundle = [(f"{r}/{n}", f"{os.path.basename(WORK)}/{os.path.basename(r)}/{n}") for r in runs for n in ("lp_dev.npz", "lp_test.npz", "meta.json")]
zip_dir(f"{OUT_ROOT}/{EXP_TAG}_{ARCH}_bundle.zip", bundle)       # small bundle of log-probs (all the ensemble needs)
allf = [(os.path.join(dp, f), os.path.relpath(os.path.join(dp, f), OUT_ROOT)) for dp, _, fs in os.walk(WORK) for f in fs if f != "last.pt"]
zip_dir(f"{OUT_ROOT}/{EXP_TAG}_{ARCH}_ALL_outputs.zip", allf)   # everything (best.pt, predictions, logs) for later phases
if AUTO_ENSEMBLE: FINAL = run_ensemble()                           # ensemble of every bundle uploaded so far, from any session
print("Optional: download *_ALL_outputs.zip from the Output panel if you want best.pt etc. Do NOT use Save Version / Save & Run All.")

In [ ]:
FINAL = run_ensemble()